character level model works by generating one alphabet after another - the word "four" has four character

In [ ]:
words = open("names.txt", "r").read().splitlines()

In [ ]:
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

In a bigram model, we are working with two characters at a time, the first is the character we are given and the second is the next charcter we are trying to predict.
The bigram is a very simple and weak model.
Bigrams are two characters neuron.

In [ ]:
# sample of what the bigrams look like - picking two chracters at a time sliding through the word
b ={}
for w in words:  
    #chs - characters, <S> - bigram of the start chracter, <E> - bigram of the end character
    chs = ['<S>'] + list(w) + ['<E>'] 
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1 # counting the bigrams

we are want to find out which character is most likly to follow another character, the simplest way to do this is by counting. counting how ofter each combinations occurs in the training set

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1]) # sorting the bigrams by their count in descending order

In [ ]:
import torch

In [ ]:
N = torch.zeros((27,27), dtype=torch.int32)

N is a matrix array of 0s that will be modified to contain all the possible combinations in the bigram models and the amount of time they occur. The rows will be the first words while the column will be the second words following it
Since we have the characters combination which are strings and we want to index it into an array(N) and we have to index using integers. we need a character lookup table from string to integers

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)} #creating Lookup table -string to integer mapping, +1 added so special character can have index position 0
stoi['.'] = 0 #special character
itos = {i:s for s,i in stoi.items()} #integer to string mapping - inverse of stoi 

In [ ]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1 

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# code for better visualization
plt.figure(figsize=(16,16))
plt.imshow(N.tolist(), cmap="Blues")

for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color="gray")
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color="gray")
plt.axis('off')

In [ ]:
N[0, :] #to sample from the bigram count matrix (image above)

N[0] - sampling from the count.
Now we have to convert the raw count (N) to Probability  (p).
before the converting to p, we have to convert it to float. this is because we are about to normalize the count (N).
Normalization here makes a probalility distribution that sums to 1. shows the probability for any character to be the first character of the word.

In [ ]:
p = N[0].float()
p = p / p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

In [ ]:
g = torch.Generator().manual_seed(2147483647) #for reproducibility
p = torch.rand(3, generator=g) #the generator will make the random numbers generated the same everytime
p = p/p.sum()
p
# output - tensor([0.6064, 0.3033, 0.0903]) - 0, 1, 2 probabilities

torch.multinomial(p, num_samples=20, replacement=True, generator=g). This allows us to pick a number from the probability distribution .
p is the torch tensor of probability distribution.
num_samples is the number of output.
replacement=True means when we draw an element, we can put it back into the list of eligigble indices to draw again.


In [ ]:
torch.multinomial(p, num_samples=30, replacement=True, generator=g)

P = P/P.sum(1, keepdim=True) same as P /= P.sum(1, keepdim=True).
P /= P.sum(1, keepdim=True). makes it a probability distribution to know the probability of picking each alphabet - used tensor sum function  - it follows tensorflow "broadcasting" rules. 
The 1 means it should sum by each row resulting in a  matrix shape of[27, 1] (AKA normalizing the rows).
if the 1 was replaced by "0" it would mean to sum by each column resulting in a matrix shape of [1, 27].
P[0].sum() should be 1. meaning the total distribution of all elements in that row sums up to 1 like a probabbility distribution should.

In [ ]:
P = (N+1).float() # Add one to the entire values in the matrix to avoid zero probabilities, known as model smoothing
P /= P.sum(1, keepdim=True)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

# generate multiples sample from the bigram model
for i in range(5):
    out = []
    ix = 0
    while True: #generate one sample from the model

        p = P[ix] #pick a row of P with the index pos, from the bigram count matrix - already converted to probabilities (image above)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item() #pick a probability from the probability distribution
        out.append(itos[ix]) #append the picked character (after converted from probability/integer) to the output list
        if ix == 0: #if the picked character is the special character (end of word),
            break
    print("".join(out))
    

In [ ]:
# GOAL: maximize the likelihood of the data w.r.t model parameters (statiscal modeling)
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood 
# equivalent to minimizing the average negative log likelihood

# log(a*bC) = log(a) + log(b) + log(c)

In [ ]:
# To calculate the loss function.
log_likelihood = 0.0
n = 0
for w in words[:5]:
# for w in ["andrejq"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        prob = P[ix1, ix2] 
        logprob = torch.log(prob) #to make the numbers smaller and easier to work with.
        log_likelihood += logprob #the product of all the probabilities (loss function) but has the opposite semantics saying low is bad, high is good
        n += 1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}') # if you want to see the log probabilities for each bigram
print(f'{log_likelihood=}')
nll = -log_likelihood #just the inverse of log_likelihood, sayiing low is good and high is bad
print(f'{nll=}')
print(f'{nll/n}') #done to normalize / get the average to make the number look smaller

Now lets start building the neural network

In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []  #xs - inputs, ys - targets (whats next)

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

. e
e m
m m
m a
a .


In [31]:
xs

tensor([ 0,  5, 13, 13,  1])

In [ ]:
ys

In [ ]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g) #weight of the neuron

In [ ]:
import torch.nn.functional as F
# this takes long tensor and return a tensor encoding - long meaning integer 
# read official documentation for better understanding

xenc = F.one_hot(xs, num_classes=27).float() 
#input to the network: x/one-hot encoding - converting to float because when sending inputs to the neural network, we want them to be float point numbers - 27 bc thats our dict size 


In [ ]:
# forward pass
xenc = F.one_hot(xs, num_classes=27).float() #input to the network: one-hot encoding
logits = xenc @ W #predict log-counts(logits) - mul of input by weight
counts = logits.exp() #counts, equivalent to N - makes the neg numbers pos 
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax'

In [32]:
probs.shape

torch.Size([5, 27])

Now the question is that can we optimize and find a good W such that the probabilities coming out are pretty good, the way we measure pretty good is by the loss function

In [ ]:
nlls = torch.zeros(5)
for i in range(5):
    # i-th bigram:
    x = xs[i].item() #input character index
    y = ys[i].item() #label character index
    print('----------')
    print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
    print('input to the neural net: ', x)
    print('output probabilities from the neural net: ', probs[i])
    print('label (actual next character): ', y)
    p = probs[i, y]
    print('probability assigned by the net to the correct character: ', p.item())
    logp = torch.log(p)
    print('log likelihood: ', logp.item())
    nll = -logp
    print('negative log likelihood: ', nll.item())
    nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

In [34]:
# ---------- !!! OPTIMIZATION !!!  --------

In [35]:
xs

tensor([ 0,  5, 13, 13,  1])

In [36]:
ys

tensor([ 5, 13, 13,  1,  0])

In [ ]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True) #weight of the neuron

In [ ]:
# forward pass
xenc = F.one_hot(xs, num_classes=27).float() #input to the network: one-hot encoding
logits = xenc @ W #predict log-counts(logits) - mul of input by weight
counts = logits.exp() #counts, equivalent to N - makes the neg numbers pos 
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -prob[torch.arange(5), ys].log().mean()

now we want to calculate loss, but unlike micrograd that used "mean-sqaured error" because it is doing regression, we will use "negative log likelihood" because we are doing classification

In [ ]:
# backward pass